# Phase 1 raw collection

Source: https://www.inven.co.kr/board/maple/2304/20563

`collect_main()` is idempotent for the committed `data/raw/` artifacts: if `inven_20563.html`, `view_by_type.txt`, or `view_by_date.txt` already exist, the code reads the existing files and does not fetch or overwrite them. Re-crawling requires intentionally removing the raw artifact first.


In [1]:
from pathlib import Path
import json
import re
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_dir = project_root / 'data' / 'raw'
raw_dir

WindowsPath('C:/Users/USER/Desktop/수업/데이터처리실습/final_project/data/raw')

In [2]:
from src.collect import collect_main

collect_main(str(raw_dir))

In [3]:
expected_files = [
    'inven_20563.html',
    'inven_20563_meta.json',
    'view_by_type.txt',
    'view_by_date.txt',
]

for name in expected_files:
    path = raw_dir / name
    text = path.read_text(encoding='utf-8')
    print(f'{name}: exists={path.exists()} chars={len(text)} bytes={path.stat().st_size}')

meta = json.loads((raw_dir / 'inven_20563_meta.json').read_text(encoding='utf-8'))
print('status_code:', meta.get('status_code'))
print('robots_allowed:', meta.get('robots_allowed'))

inven_20563.html: exists=True chars=629378 bytes=698829
inven_20563_meta.json: exists=True chars=417 bytes=426
view_by_type.txt: exists=True chars=23476 bytes=31110
view_by_date.txt: exists=True chars=43800 bytes=82038
status_code: 200
robots_allowed: True


In [4]:
for name in ['view_by_type.txt', 'view_by_date.txt']:
    text = (raw_dir / name).read_text(encoding='utf-8')
    date_count = len(re.findall(r'\d{4}\.\d{2}\.\d{2}', text))
    print(f'{name}: date_count={date_count} B_markers={text.count("[[B]]")} S_markers={text.count("[[S]]")}')

view_by_type.txt: date_count=1140 B_markers=71 S_markers=0
view_by_date.txt: date_count=503 B_markers=568 S_markers=2


In [5]:
for name in ['view_by_type.txt', 'view_by_date.txt']:
    print(f'--- {name} first 30 lines ---')
    lines = (raw_dir / name).read_text(encoding='utf-8').splitlines()[:30]
    print('\n'.join(lines))

--- view_by_type.txt first 30 lines ---
- [HEXA 스탯] 메인 스탯이 5레벨 이상일 때, 메인 스탯의 강화 확률 20% 증가
2024.07.21
(+84일)2024.10.13
(+91일)2025.01.12
(+84일)2025.04.06
(+91일)2025.07.06
(+112일)2025.10.26
(+77일)2026.01.11
(+98일)2026.04.19
- 10성 이하에서 스타포스 강화 성공 시 1+1 강화
2018.02.18
(+98일)2018.05.27
(+49일)2018.07.15
(+140일)2018.12.02
(+63일)2019.02.03
(+98일)2019.05.12
(+49일)2019.06.30
(+63일)2019.09.01
(+119일)2019.12.29
(+98일)2020.04.05
(+77일)2020.06.21
(+42일)2020.08.02
(+77일)2020.10.18
(+126일)2021.02.21
(+63일)2021.04.25
(+49일)2021.06.13
(+49일)2021.08.01
(+49일)2021.09.19
(+49일)2021.11.07
(+154일)2022.04.10
--- view_by_date.txt first 30 lines ---
2017.03.12 (-)
주문의 흔적 강화 비용 30% 할인
(리부트, 리부트2월드 : 하루 종일 드롭률 2배)
돌발미션 일일 클리어 가능 횟수 2배(3회→6회)
돌발미션 클리어 경험치 2배
[[S]]2017.03.19[[/S]]
[[S]]아예 없음[[/S]]
2017.03.26 (도전! 메이플 레인보우!)
스타포스 5, 10, 15성에서 강화 시도 시 성공확률 100%
룬 경험치 버프 효과 +100%
룬 경험치 버프 지속시간 2배(3분→6분)
2017.04.02 (달콤한 질주 : 더 꿀)
아케인리버 1~6지역 일일 컨텐츠 획득 보상 2배
폴로와 프리토의 현상금 사냥 경험치 1.5배
2017.04.09 (달콤한 질주 : 더 꿀)
어빌리티 재설정 비용 5

## Reproducibility notes

- `data/raw/` is committed and treated as immutable for downstream phases.
- `save_raw_html()` returns existing metadata when the HTML file is present.
- `collect_main()` only writes extracted view files when their target paths do not already exist.
- Verified workflow expectation: rerunning 01 with committed raw files should leave `git status data/raw/` unchanged.
